# DeepClean

* Missing Values
* Outliers
* Categorical Encoding
* Date Features
* Normalization / Standardization
* Duplicates

In [3]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from datetime import datetime, timedelta
import random
from scipy import stats

print("Libraries imported successfully!")

Libraries imported successfully!


## Data Generation Function

Uses vectorized numpy operations to ensure it can scale to millions of rows quickly without relying on slow pandas `.apply()` methods during the base generation.

In [4]:
def generate_mock_data(n_rows=1_000_000):
    print(f"Generating {n_rows} base rows...")
    
    # 1. Base Structure
    np.random.seed(42)
    ids = np.arange(1, n_rows + 1)
    
    # 2. Date Features (Random dates over a 5-year period)
    base_date = np.datetime64('2020-01-01')
    random_days = np.random.randint(0, 1825, size=n_rows)
    dates = base_date + random_days.astype('timedelta64[D]')
    
    # 3. Categorical Encoding (Low and High Cardinality)
    categories = ['Electronics', 'Clothing', 'Home', 'Toys', 'Sports']
    cat_probs = [0.4, 0.25, 0.15, 0.1, 0.1]
    departments = np.random.choice(categories, size=n_rows, p=cat_probs)
    
    # 4. Normalization / Standardization (Normal vs Skewed)
    temperature = np.random.normal(loc=22.0, scale=5.0, size=n_rows) 
    revenue = np.random.exponential(scale=150.0, size=n_rows) 
    
    # Construct initial DataFrame
    df = pd.DataFrame({
        'transaction_id': ids,
        'date_recorded': dates,
        'department': departments,
        'temperature_celsius': temperature,
        'revenue': revenue
    })

    print("Injecting dirty data (Missing values, Outliers, Messy strings, Duplicates)...")

    # --- INJECTING MISSING VALUES ---
    dept_nan_idx = np.random.choice(df.index, size=int(n_rows * 0.10), replace=False)
    temp_nan_idx = np.random.choice(df.index, size=int(n_rows * 0.05), replace=False)
    df.loc[dept_nan_idx, 'department'] = np.nan
    df.loc[temp_nan_idx, 'temperature_celsius'] = np.nan

    # --- INJECTING OUTLIERS ---
    outlier_idx = np.random.choice(df.index, size=int(n_rows * 0.01), replace=False)
    df.loc[outlier_idx, 'revenue'] = df.loc[outlier_idx, 'revenue'] * np.random.uniform(100, 1000, size=len(outlier_idx))
    
    neg_idx = np.random.choice(df.index, size=int(n_rows * 0.005), replace=False)
    df.loc[neg_idx, 'revenue'] = df.loc[neg_idx, 'revenue'] * -1

    # --- INJECTING MESSY CATEGORICALS ---
    messy_idx = np.random.choice(df.dropna(subset=['department']).index, size=int(n_rows * 0.05), replace=False)
    df.loc[messy_idx, 'department'] = df.loc[messy_idx, 'department'].apply(
        lambda x: str(x).upper() + "   " if np.random.rand() > 0.5 else "  " + str(x).lower()
    )

    # --- INJECTING DUPLICATES ---
    duplicates = df.sample(frac=0.05, random_state=42)
    df = pd.concat([df, duplicates], ignore_index=True)

    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"Data generation complete. Final shape: {df.shape}")
    return df

## Execute Generator

**Note on Memory Constraints:** Adjust `n_rows` carefully. If your machine freezes, it is likely due to the Jupyter kernel retaining data copies in memory. Use `gc.collect()` if doing multiple runs.

In [5]:
df_dirty = generate_mock_data(n_rows=1_000_000)

Generating 1000000 base rows...
Injecting dirty data (Missing values, Outliers, Messy strings, Duplicates)...
Data generation complete. Final shape: (1050000, 5)


## Verify Targets

In [6]:
df_dirty.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1050000 entries, 0 to 1049999
Data columns (total 5 columns):
 #   Column               Non-Null Count    Dtype        
---  ------               --------------    -----        
 0   transaction_id       1050000 non-null  int64        
 1   date_recorded        1050000 non-null  datetime64[s]
 2   department           945019 non-null   object       
 3   temperature_celsius  997506 non-null   float64      
 4   revenue              1050000 non-null  float64      
dtypes: datetime64[s](1), float64(2), int64(1), object(1)
memory usage: 40.1+ MB


In [7]:
df_dirty.isnull().sum()

transaction_id              0
date_recorded               0
department             104981
temperature_celsius     52494
revenue                     0
dtype: int64